# Colab 环境冒烟测试 · Skincare Advisor

**目的**:在花钱做教师蒸馏、在正式训练之前,先确认 Colab 这条路走得通。

**这个 notebook 不训练有用的模型** —— 它只回答一个问题:*我的代码能在 Colab 上跑起来吗?*

| 检查项 | 失败意味着 |
|---|---|
| 1. GPU | 没选 GPU runtime,或分到了不支持的卡 |
| 2. 代码就位 | 上传/clone 有问题 |
| 3. 依赖安装 | Colab 版本冲突(**最常见的坑**) |
| 4. TRL API 兼容 | 库升级了,训练脚本要改参数名 |
| 5. 测试套件 | 代码本身有问题 |
| 6. 造数据 | 数据管线有问题 |
| 7. 基座模型加载 | 显存不够,要换小模型 |
| 8. SFT 冒烟 | **训练管线通不通,这是关键** |
| 9. GRPO 冒烟 | **RL 管线通不通,这是关键** |

**预计 20–30 分钟**(大部分时间在下模型)。全绿之后,再去做教师蒸馏和正式训练。

> ⚠️ 先确认:菜单 `代码执行程序 → 更改运行时类型 → T4 GPU`


## 1. GPU 检查


In [ ]:
import torch, subprocess, shutil

if shutil.which('nvidia-smi'):
    out = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip()
    print('GPU:', out)
else:
    print('GPU: 检测不到 nvidia-smi')

print('torch:', torch.__version__, '| CUDA 可用:', torch.cuda.is_available())

if not torch.cuda.is_available():
    print('\n❌ 没有 GPU。菜单 -> 代码执行程序 -> 更改运行时类型 -> T4 GPU,然后重跑本格')
else:
    bf16 = torch.cuda.is_bf16_supported()
    print(f'bf16 支持: {bf16}  ->  训练将使用 {"bf16" if bf16 else "fp16"}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print('\n✅ 第 1 步通过')


## 2. 把代码放上来

**两种方式,二选一:**

- **有 GitHub 仓库**:把地址填进 `REPO_URL`
- **还没推 GitHub**:留空,会让你上传 zip。先在 Mac 上打包:

```bash
cd ~/Documents && zip -r skincare.zip skincare -x '*.git*' '*__pycache__*' '*.venv*'
```


In [ ]:
REPO_URL = ''   # 例如 'https://github.com/yourname/skincare.git';留空则上传 zip

import os, sys, shutil, zipfile
from pathlib import Path

WORK = Path('/content/skincare')
if WORK.exists(): shutil.rmtree(WORK)

if REPO_URL:
    os.system(f'git clone -q {REPO_URL} {WORK}')
else:
    from google.colab import files
    print('请上传 skincare.zip …')
    up = files.upload()
    name = list(up)[0]
    tmp = Path('/content/_unzip')
    if tmp.exists(): shutil.rmtree(tmp)
    with zipfile.ZipFile(name) as z: z.extractall(tmp)
    root = next(p.parent for p in tmp.rglob('pyproject.toml'))
    shutil.move(str(root), str(WORK))

os.chdir(WORK)
sys.path[:0] = [str(WORK), str(WORK/'src')]
os.environ['PYTHONPATH'] = f"{WORK}:{WORK/'src'}"

n_py = len(list(WORK.rglob('*.py')))
ok = (WORK/'src'/'skincare'/'llm'/'grpo_train.py').exists()
print(f'目录: {WORK}   Python 文件: {n_py}')
print('✅ 第 2 步通过' if ok else '❌ 找不到 grpo_train.py,检查压缩包结构')


## 3. 安装依赖

Colab 自带 torch。这里装训练相关的库。**如果提示需要重启运行时,重启后从第 2 格重跑。**


In [ ]:
import os, importlib, importlib.util

# ---- 1) 装训练依赖 ----
os.system('pip install -q transformers peft trl datasets accelerate 2>&1 | tail -3')
os.system('pip install -q fastapi pydantic python-multipart python-dotenv pandas pytest httpx pyyaml 2>&1 | tail -2')

# ---- 2) 处理 Colab 自带的旧版 torchao ----
# Colab 预装 torchao 0.10,而新版 PEFT 要求 >0.16:它在探测阶段会直接 raise ImportError,
# 导致 LoRA 一初始化就崩(报错出现在 peft/tuners/lora/torchao.py)。
# 我们做的是纯 fp16 LoRA,不涉及量化,用不到 torchao —— 卸掉最干净。
if importlib.util.find_spec('torchao') is not None:
    try:
        import importlib.metadata as _md
        v = _md.version('torchao')
    except Exception:
        v = '未知'
    print(f'检测到 torchao {v} —— 卸载以避免 PEFT 冲突')
    os.system('pip uninstall -y -q torchao')
    importlib.invalidate_caches()
else:
    print('torchao 未安装,无需处理')

# ---- 3) 报告版本 ----
print()
vers = {}
for m in ['torch','transformers','peft','trl','datasets','accelerate']:
    try:
        vers[m] = importlib.import_module(m).__version__
    except Exception as e:
        vers[m] = f'❌ {e}'
for k, v in vers.items(): print(f'{k:14s} {v}')

torchao_gone = importlib.util.find_spec('torchao') is None
print(f"\ntorchao 已清除: {torchao_gone}")
ok = all('❌' not in str(v) for v in vers.values()) and torchao_gone
print('\n✅ 第 3 步通过' if ok else '\n❌ 见上方问题')


### 常见问题

**`ImportError: Found an incompatible version of torchao`(第 8 格 SFT 报错)**
上一格已自动处理。若仍出现,手动跑:`!pip uninstall -y torchao`,然后重跑第 8 格。
原因:Colab 预装 torchao 0.10,新版 PEFT 要求 >0.16,探测时直接抛异常。我们不用量化,卸掉即可。

**提示「需要重启运行时」**
点重启,然后从第 2 格重新往下跑。

**第 7 或第 8 格 OOM**
把第 7 格的 `BASE` 改成 `Qwen/Qwen2.5-0.5B-Instruct`,重跑第 7 格起。

**第 4 格报 TRL 参数缺失**
TRL 又改 API 了。把缺失的参数名发给 Claude 改代码,或临时 `!pip install -q "trl==0.21.0"` 后重启运行时。


## 4. TRL API 兼容性检查 ⚠️ 最容易出问题的一步

TRL 在版本之间会改参数名(例如 `max_seq_length` → `max_length`,`max_prompt_length` 被移除)。
这里逐个核对训练脚本用到的参数是否还存在——**比直接跑训练再看报错快得多**。


In [ ]:
import dataclasses
from trl import SFTConfig, GRPOConfig

NEED = {
    'SFTConfig': (SFTConfig, ['max_length','num_train_epochs','per_device_train_batch_size',
                              'gradient_accumulation_steps','learning_rate','logging_steps',
                              'save_strategy','bf16','fp16','gradient_checkpointing',
                              'report_to','output_dir']),
    'GRPOConfig': (GRPOConfig, ['num_generations','max_steps','per_device_train_batch_size',
                                'gradient_accumulation_steps','learning_rate','max_completion_length',
                                'beta','temperature','logging_steps','save_steps','bf16','fp16',
                                'report_to','output_dir']),
}
all_ok = True
for name, (cls, need) in NEED.items():
    have = {f.name for f in dataclasses.fields(cls)}
    miss = [n for n in need if n not in have]
    all_ok &= not miss
    print(f'{name}: ' + ('✅ 参数齐全' if not miss else f'❌ 缺少 {miss}'))

if all_ok:
    print('\n✅ 第 4 步通过 —— TRL API 与训练脚本兼容')
else:
    print('\n❌ TRL 又改 API 了。把上面缺失的参数名发给 Claude,改完再跑。')
    print('   临时办法:!pip install -q "trl==0.21.0" 然后重启运行时')


## 5. 测试套件(27 个)


In [ ]:
!python -m pytest tests/ -q 2>&1 | tail -6
print('\n↑ 应该看到 27 passed。有失败先别往下走。')


## 6. 造数据(零成本)

用合成产品目录 + 离线假教师 —— **不调用 OpenAI,不花钱**,只为验证数据管线。


In [ ]:
!python -m skincare.llm.data_build --n 40 --mock-retrieval --dry-teacher 2>&1 | tail -10

from pathlib import Path
need = ['sft.jsonl','rl.jsonl','rl_test.jsonl']
got = [f for f in need if (Path('data/processed')/f).exists()]
print(f'\n生成的文件: {got}')
print('✅ 第 6 步通过' if len(got)==3 else f'❌ 缺少 {set(need)-set(got)}')


## 7. 加载基座模型 + 显存检查

T4(16GB)跑 1.5B 应该没问题。**如果 OOM,把 `BASE` 换成 `Qwen/Qwen2.5-0.5B-Instruct`。**


In [ ]:
BASE = 'Qwen/Qwen2.5-1.5B-Instruct'   # OOM 就换成 Qwen/Qwen2.5-0.5B-Instruct
import os; os.environ['BASE'] = BASE

import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    tok = AutoTokenizer.from_pretrained(BASE)
    m = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map='auto')
    n = sum(p.numel() for p in m.parameters())/1e9
    used = torch.cuda.memory_allocated()/1e9
    total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'✅ 加载成功  参数 {n:.2f}B  权重占显存 {used:.2f}/{total:.1f} GB')
    print(f'   训练还要额外占激活值与优化器状态,剩余 {total-used:.1f} GB 应当够 LoRA 用')
    del m; gc.collect(); torch.cuda.empty_cache()
    print('\n✅ 第 7 步通过')
except Exception as e:
    print(f'❌ 加载失败: {type(e).__name__}: {e}')
    print('   OOM 的话把 BASE 换成 Qwen/Qwen2.5-0.5B-Instruct 重跑本格')


## 8. SFT 冒烟测试 🔑

**这是关键一步** —— 真的用 LoRA 微调基座模型跑几步。数据是假教师产出的,
所以**模型本身没有价值**,我们只看:管线通不通、显存够不够、速度多快。


In [ ]:
!python -m skincare.llm.sft_lora --base $BASE --epochs 1 --bs 1 --accum 4 \
    --max-len 1024 --out /content/smoke-sft 2>&1 | tail -12

from pathlib import Path
ok = (Path('/content/smoke-sft')/'adapter_model.safetensors').exists() or \
     any(Path('/content/smoke-sft').glob('adapter*'))
print('\n✅ 第 8 步通过 —— SFT 管线可用' if ok else '\n❌ SFT 没产出 adapter,看上面报错')


## 9. GRPO 冒烟测试 🔑

从上一步的 SFT adapter 继续做 RL 后训练,跑 3 步。

**重点看输出里的 `rewards/xxx_reward/mean`** —— 五个奖励分量都要出现,
这说明 TRL 正确识别了奖励函数。数值低是正常的(模型只训了几步)。


In [ ]:
!python -m skincare.llm.grpo_train --base $BASE --adapter /content/smoke-sft \
    --steps 3 --group-size 4 --accum 1 --max-completion-length 128 \
    --out /content/smoke-grpo 2>&1 | tail -14

from pathlib import Path
ok = any(Path('/content/smoke-grpo').glob('adapter*'))
print('\n✅ 第 9 步通过 —— GRPO 管线可用' if ok else '\n❌ GRPO 没产出 adapter,看上面报错')


## 10. 结论


In [ ]:
from pathlib import Path
import torch

checks = {
    '1 GPU 可用': torch.cuda.is_available(),
    '2 代码就位': Path('src/skincare/llm/grpo_train.py').exists(),
    '6 数据生成': Path('data/processed/sft.jsonl').exists(),
    '8 SFT 管线': any(Path('/content/smoke-sft').glob('adapter*')),
    '9 GRPO 管线': any(Path('/content/smoke-grpo').glob('adapter*')),
}
for k, v in checks.items():
    print(f'  {"✅" if v else "❌"}  {k}')

if all(checks.values()):
    print('\n' + '='*52)
    print('  全绿 —— Colab 这条路走得通,可以进入下一步了')
    print('='*52)
    print('\n下一步(按顺序):')
    print('  1) 小样本教师蒸馏(约几分钱),先看质量与过滤通过率:')
    print('     export OPENAI_API_KEY=sk-...')
    print('     python -m skincare.llm.data_build --n 20 --mock-retrieval --mode sft')
    print('     -> 肉眼读几条 sft.jsonl,并看"保留/丢弃"比例')
    print('  2) 通过率高(>70%)再上量: --n 800')
    print('  3) 正式 SFT: --epochs 2')
    print('  4) 正式 GRPO: --steps 300')
else:
    print('\n有失败项,把对应格子的报错发给 Claude。')
